In [ ]:
#NFNet Model
import os, copy, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/AML_Dataset/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!pip install timm -q
import timm

Using device: cuda
Mounted at /content/drive


In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile
with zipfile.ZipFile("WaRP-C-preprocessed.zip", "r") as z:
    z.extractall("/content/WaRP-C-preprocessed")

Saving WaRP-C-preprocessed.zip to WaRP-C-preprocessed.zip


In [ ]:
PREPROCESSED_ROOT = "/content/WaRP-C-preprocessed"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CFG = {
    "num_classes":28,
    "model_name":"dm_nfnet_f0",   # NFNet-F0, ~71M params, supervised ImageNet-pretrained
    "lr": 3e-4,
    "min_lr":1e-6,
    "weight_decay":0.05,
    "label_smoothing":0.1,
    "focal_gamma":2.0,
    "warmup_epochs":3,
    "num_epochs":15,
    "blend_alpha":0.4,
    "early_stop_patience":5,
}

full_train_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

eval_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def _make_flat_dataset(root_dir, transform):
    samples, class_to_idx = [], {}
    for superclass in sorted(os.listdir(root_dir)):
        sp = os.path.join(root_dir, superclass)
        if not os.path.isdir(sp): continue
        for subclass in sorted(os.listdir(sp)):
            scp = os.path.join(sp, subclass)
            if not os.path.isdir(scp): continue
            if subclass not in class_to_idx:
                class_to_idx[subclass] = len(class_to_idx)
            for img_name in os.listdir(scp):
                if img_name.lower().endswith(".jpg"):
                    samples.append((os.path.join(scp, img_name), class_to_idx[subclass]))
    dataset = datasets.ImageFolder(root_dir, transform=transform)
    dataset.samples = dataset.imgs = samples
    dataset.targets = [s[1] for s in samples]
    dataset.classes = list(class_to_idx.keys())
    dataset.class_to_idx = class_to_idx
    return dataset


def get_dataloaders(root=PREPROCESSED_ROOT, batch_size=32, num_workers=2, seed=42):
    torch.manual_seed(seed)
    train_ds = _make_flat_dataset(f"{root}/train", transform=full_train_pipeline)
    val_ds   = _make_flat_dataset(f"{root}/val",   transform=eval_pipeline)
    test_ds  = _make_flat_dataset(f"{root}/test",  transform=eval_pipeline)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader= DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    test_loader= DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


def get_class_counts(dataset):
    counts = np.bincount(dataset.targets, minlength=len(dataset.classes))
    return counts.astype(np.float32)


train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders()
num_classes = len(train_loader.dataset.classes)
print(f"Classes: {num_classes} | Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Classes: 28 | Train: 7058 | Val: 1765 | Test: 1551


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits, targets, weight=self.weight,
            label_smoothing=self.label_smoothing, reduction="none"
        )
        pt = torch.exp(-ce_loss)
        focal_term = (1 - pt) ** self.gamma
        return (focal_term * ce_loss).mean()


def get_criterion(loss_type, samples_per_class=None, device=None, label_smoothing=0.1, gamma=2.0):
    weight = None
    if loss_type in ("weighted_ce", "focal") and samples_per_class is not None:
        freq_inverse = 1.0 / (samples_per_class + 1e-6)
        weight = torch.tensor(freq_inverse / freq_inverse.sum(), dtype=torch.float).to(device)

    if loss_type == "ce":
        return nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    elif loss_type == "weighted_ce":
        return nn.CrossEntropyLoss(weight=weight, label_smoothing=label_smoothing)
    elif loss_type == "focal":
        return FocalLoss(weight=weight, gamma=gamma, label_smoothing=label_smoothing)
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

In [ ]:
def build_nfnet(num_classes, freeze_backbone=False):
    model = timm.create_model(CFG["model_name"], pretrained=True, num_classes=num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            if "head" not in name and "fc" not in name and "classifier" not in name:
                param.requires_grad = False
    return model


def get_param_groups(model):
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "head" in name or "fc" in name or "classifier" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
    return backbone_params, head_params

In [ ]:
def mixup_data(x, y, alpha=1.0):
    blend_ratio = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    blended_imgs = blend_ratio * x + (1 - blend_ratio) * x[index, :]
    return blended_imgs, y, y[index], blend_ratio

def mixup_criterion(criterion, pred, labels_orig, labels_mixed, blend_ratio):
    return blend_ratio * criterion(pred, labels_orig) + (1 - blend_ratio) * criterion(pred, labels_mixed)

def train_one_epoch(model, loader, optimizer, criterion, blend_alpha=0.0):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if blend_alpha > 0:
            imgs, labels_orig, labels_mixed, blend_ratio = mixup_data(imgs, labels, blend_alpha)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = mixup_criterion(criterion, logits, labels_orig, labels_mixed, blend_ratio) if blend_alpha > 0 else criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, eval_device=None):
    eval_device = eval_device or device
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(eval_device), labels.to(eval_device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

In [ ]:
def run_nfnet_experiment(freeze_backbone, loss_type):
    config_name = f"nfnet_frozen{freeze_backbone}_loss{loss_type}"
    checkpoint_path = f"{CHECKPOINT_DIR}/{config_name}_checkpoint.pth"

    print(f"CONFIG: {config_name}")

    model = build_nfnet(num_classes, freeze_backbone=freeze_backbone).to(device)

    samples_per_class = get_class_counts(train_ds)
    criterion = get_criterion(loss_type, samples_per_class=samples_per_class, device=device,
                               label_smoothing=CFG["label_smoothing"], gamma=CFG["focal_gamma"])

    backbone_params, head_params = get_param_groups(model)
    if freeze_backbone:
        optimizer = optim.AdamW(head_params, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    else:
        optimizer = optim.AdamW([
            {"params": backbone_params, "lr": CFG["lr"] / 10},
            {"params": head_params, "lr": CFG["lr"]},
        ], weight_decay=CFG["weight_decay"])

    def get_lr_scale(epoch):
        if epoch < CFG["warmup_epochs"]:
            return (epoch + 1) / CFG["warmup_epochs"]
        decay_progress = (epoch - CFG["warmup_epochs"]) / max(1, CFG["num_epochs"] - CFG["warmup_epochs"])
        return CFG["min_lr"] / CFG["lr"] + 0.5 * (1 - CFG["min_lr"] / CFG["lr"]) * (1 + np.cos(np.pi * decay_progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, get_lr_scale)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, patience_counter, start_epoch = 0.0, 0, 0

    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt["best_val_acc"]
        patience_counter = ckpt["patience_counter"]
        history = ckpt["history"]

    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(start_epoch, CFG["num_epochs"]):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, blend_alpha=CFG["blend_alpha"])
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch+1:02d}/{CFG['num_epochs']}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"  New best val acc: {best_val_acc:.4f}")
        else:
            patience_counter += 1

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "best_model_state": best_state,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "patience_counter": patience_counter,
            "history": history,
            "cfg": CFG,
        }, checkpoint_path)

        if patience_counter >= CFG["early_stop_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="weighted", zero_division=0)
    cm = confusion_matrix(test_labels, test_preds)

    result = {
        "freeze_backbone": freeze_backbone,
        "loss_type": loss_type,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "history": history,
    }

    with open(f"{config_name}_results.json", "w") as f:
        json.dump({k: v for k, v in result.items()}, f, indent=2)
    np.savez(f"{config_name}_test_predictions.npz",
             preds=np.array(test_preds), labels=np.array(test_labels),
             class_names=np.array(train_loader.dataset.classes))
    torch.save(model.state_dict(), f"{config_name}_final.pth")

    from google.colab import files
    files.download(f"{config_name}_results.json")
    files.download(f"{config_name}_test_predictions.npz")
    files.download(f"{config_name}_final.pth")

    return result, model, criterion, test_loader, cm

In [ ]:
#Ablation Study- Frozen backbone+class-weighted loss
result_r1, model_r1, criterion_r1, test_loader_r1, cm_r1 = run_nfnet_experiment(freeze_backbone=True, loss_type="weighted_ce")
print(result_r1)

CONFIG: nfnet_frozenTrue_lossweighted_ce


model.safetensors: reconstructing file:   0%|          |  0.00B /  286MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [03/15] Train Loss: 2.4580 Acc: 0.303 | Val Loss: 2.6334 Acc: 0.583 | LR: 3.00e-04
  New best val acc: 0.5830
Epoch [04/15] Train Loss: 2.3529 Acc: 0.313 | Val Loss: 2.5450 Acc: 0.614 | LR: 2.95e-04
  New best val acc: 0.6136
Epoch [05/15] Train Loss: 2.3101 Acc: 0.321 | Val Loss: 2.5078 Acc: 0.640 | LR: 2.80e-04
  New best val acc: 0.6397
Epoch [06/15] Train Loss: 2.2561 Acc: 0.375 | Val Loss: 2.4346 Acc: 0.678 | LR: 2.56e-04
  New best val acc: 0.6782
Epoch [07/15] Train Loss: 2.2128 Acc: 0.384 | Val Loss: 2.4058 Acc: 0.677 | LR: 2.25e-04
Epoch [08/15] Train Loss: 2.1105 Acc: 0.394 | Val Loss: 2.3755 Acc: 0.705 | LR: 1.89e-04
  New best val acc: 0.7054
Epoch [09/15] Train Loss: 2.0647 Acc: 0.388 | Val Loss: 2.3661 Acc: 0.719 | LR: 1.50e-04
  New best val acc: 0.7190
Epoch [10/15] Train Loss: 2.0361 Acc: 0.393 | Val Loss: 2.3849 Acc: 0.696 | LR: 1.12e-04
Epoch [11/15] Train Loss: 2.0004 Acc: 0.421 | Val Loss: 2.3692 Acc: 0.713 | LR: 7.58e-05
Epoch [12/15] Train Loss: 1.9654 Acc:

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'freeze_backbone': True, 'loss_type': 'weighted_ce', 'best_val_acc': 0.7314447592067989, 'test_acc': 0.7143778207607995, 'precision': 0.7391075298413641, 'recall': 0.7143778207607995, 'f1': 0.7160554117480661, 'history': {'train_loss': [3.079476353255185, 2.596411191333424, 2.458020718531175, 2.352915210615505, 2.310090923309326, 2.2560739918188615, 2.2128049796277827, 2.110505868629976, 2.0646543101830916, 2.036107835444537, 2.0003962110389364, 1.9653622581200165, 1.9521572064269672, 1.9269694956866177, 1.8952321835539558], 'val_loss': [2.8721229054772484, 2.7310215841947127, 2.6334032790856727, 2.544977872027216, 2.5077739242791455, 2.434579215711642, 2.405819786979524, 2.375481722375151, 2.3661411661601943, 2.384880342159325, 2.369191622936692, 2.371980513594306, 2.327005080393286, 2.32775110570297, 2.3158295972489094], 'train_acc': [0.16860795454545455, 0.2778409090909091, 0.30326704545454547, 0.31349431818181817, 0.32073863636363636, 0.37485795454545456, 0.38423295454545453, 0.39

In [ ]:
#Ablation Study- fine tuned backbone backbone+class-weighted loss
result_r2, model_r2, criterion_r2, test_loader_r2, cm_r2 = run_nfnet_experiment(freeze_backbone=False, loss_type="weighted_ce")
print(result_r2)

CONFIG: nfnet_frozenFalse_lossweighted_ce
Epoch [01/15] Train Loss: 2.9692 Acc: 0.173 | Val Loss: 2.7650 Acc: 0.530 | LR: 2.00e-05
  New best val acc: 0.5303
Epoch [02/15] Train Loss: 2.4919 Acc: 0.277 | Val Loss: 2.5783 Acc: 0.625 | LR: 3.00e-05
  New best val acc: 0.6255
Epoch [03/15] Train Loss: 2.3798 Acc: 0.337 | Val Loss: 2.5240 Acc: 0.642 | LR: 3.00e-05
  New best val acc: 0.6425
Epoch [04/15] Train Loss: 2.3128 Acc: 0.341 | Val Loss: 2.3554 Acc: 0.727 | LR: 2.95e-05
  New best val acc: 0.7275
Epoch [05/15] Train Loss: 2.1508 Acc: 0.377 | Val Loss: 2.3550 Acc: 0.701 | LR: 2.80e-05
Epoch [06/15] Train Loss: 2.1298 Acc: 0.376 | Val Loss: 2.3113 Acc: 0.724 | LR: 2.56e-05
Epoch [07/15] Train Loss: 1.9436 Acc: 0.404 | Val Loss: 2.3276 Acc: 0.718 | LR: 2.25e-05
Epoch [08/15] Train Loss: 1.9690 Acc: 0.423 | Val Loss: 2.2886 Acc: 0.746 | LR: 1.89e-05
  New best val acc: 0.7462
Epoch [09/15] Train Loss: 1.9059 Acc: 0.456 | Val Loss: 2.1952 Acc: 0.786 | LR: 1.50e-05
  New best val acc: 0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'freeze_backbone': False, 'loss_type': 'weighted_ce', 'best_val_acc': 0.8, 'test_acc': 0.7956157317859446, 'precision': 0.8023489059695607, 'recall': 0.7956157317859446, 'f1': 0.795774297105343, 'history': {'train_loss': [2.969247249581597, 2.4918758896264164, 2.379791536656293, 2.312805914878845, 2.1507803911512546, 2.1298313801938837, 1.943581183661114, 1.9690319692546672, 1.9058643761006269, 1.8982672902670774, 1.7728499751199376, 1.7980922555381602, 1.792479285326871, 1.7170237961140546, 1.7134079109538685], 'val_loss': [2.7650093868839165, 2.5783354050039233, 2.523959820398866, 2.3553863564564215, 2.3549868291903486, 2.311322069168091, 2.327631519468918, 2.2885941188666368, 2.195180395574813, 2.2250090747649542, 2.2422575410297165, 2.167792455670516, 2.163720488075494, 2.1658458457114675, 2.1571547276575234], 'train_acc': [0.17301136363636363, 0.2768465909090909, 0.33735795454545453, 0.34133522727272725, 0.37670454545454546, 0.37585227272727273, 0.40397727272727274, 0.42315340909

In [ ]:
print("\n SUMMARY")
for r in [result_r1, result_r2]:
    print(f"frozen={r['freeze_backbone']:<5} loss={r['loss_type']:<12} "
          f"| acc={r['test_acc']:.4f} prec={r['precision']:.4f} rec={r['recall']:.4f} f1={r['f1']:.4f}")


 SUMMARY
frozen=1     loss=weighted_ce  | acc=0.7144 prec=0.7391 rec=0.7144 f1=0.7161
frozen=0     loss=weighted_ce  | acc=0.7956 prec=0.8023 rec=0.7956 f1=0.7958


In [ ]:
import torch.quantization
#Quantization
fp32_model_cpu = copy.deepcopy(model_r2).to("cpu").eval()
quantized_model = torch.quantization.quantize_dynamic(fp32_model_cpu, {nn.Linear}, dtype=torch.qint8)

def get_model_size_mb(m, filename="temp_model.pth"):
    torch.save(m.state_dict(), filename)
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    os.remove(filename)
    return size_mb

fp32_size = get_model_size_mb(fp32_model_cpu)
int8_size = get_model_size_mb(quantized_model)
print(f"FP32 size: {fp32_size:.2f} MB | INT8 size: {int8_size:.2f} MB | Compression: {fp32_size/int8_size:.2f}x")

cpu = torch.device("cpu")
criterion_r2 = criterion_r2.to(cpu)

fp32_loss, fp32_acc, fp32_preds, fp32_labels = evaluate(fp32_model_cpu, test_loader_r2, criterion_r2, eval_device=cpu)
fp32_prec, fp32_rec, fp32_f1, _ = precision_recall_fscore_support(fp32_labels, fp32_preds, average="weighted", zero_division=0)

int8_loss, int8_acc, int8_preds, int8_labels = evaluate(quantized_model, test_loader_r2, criterion_r2, eval_device=cpu)
int8_prec, int8_rec, int8_f1, _ = precision_recall_fscore_support(int8_labels, int8_preds, average="weighted", zero_division=0)

print(f"FP32 (CPU) -> Acc: {fp32_acc:.4f} | Prec: {fp32_prec:.4f} | Rec: {fp32_rec:.4f} | F1: {fp32_f1:.4f}")
print(f"INT8 (CPU) -> Acc: {int8_acc:.4f} | Prec: {int8_prec:.4f} | Rec: {int8_rec:.4f} | F1: {int8_f1:.4f}")

def measure_latency(m, input_size=(1, 3, 224, 224), n_runs=50, warmup=5):
    m.eval()
    dummy_input = torch.randn(input_size)
    with torch.no_grad():
        for _ in range(warmup):
            _ = m(dummy_input)
        start = time.time()
        for _ in range(n_runs):
            _ = m(dummy_input)
        elapsed = time.time() - start
    return (elapsed / n_runs) * 1000

fp32_latency_ms = measure_latency(fp32_model_cpu)
int8_latency_ms = measure_latency(quantized_model)
print(f"FP32 CPU latency: {fp32_latency_ms:.2f} ms/img | INT8 CPU latency: {int8_latency_ms:.2f} ms/img | Speedup: {fp32_latency_ms/int8_latency_ms:.2f}x")

quant_results = {
    "fp32_size_mb": fp32_size, "int8_size_mb": int8_size,
    "fp32_acc": fp32_acc, "int8_acc": int8_acc, "fp32_f1": fp32_f1, "int8_f1": int8_f1,
    "fp32_latency_ms": fp32_latency_ms, "int8_latency_ms": int8_latency_ms,
}
with open("nfnet_quantization_results.json", "w") as f:
    json.dump(quant_results, f, indent=2)
torch.save(quantized_model.state_dict(), "nfnet_int8.pth")

from google.colab import files
files.download("nfnet_quantization_results.json")
files.download("nfnet_int8.pth")

/tmp/ipykernel_809/3111725004.py:4: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(fp32_model_cpu, {nn.Linear}, dtype=torch.qint8)


FP32 size: 261.39 MB | INT8 size: 261.15 MB | Compression: 1.00x
FP32 (CPU) -> Acc: 0.7956 | Prec: 0.8023 | Rec: 0.7956 | F1: 0.7958
INT8 (CPU) -> Acc: 0.7956 | Prec: 0.8022 | Rec: 0.7956 | F1: 0.7958
FP32 CPU latency: 445.84 ms/img | INT8 CPU latency: 463.58 ms/img | Speedup: 0.96x


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>